# 26. Explainable Retrieval

Show *why* each passage was retrieved and cite sources in the answer.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Users trust answers they can verify. **Explainable retrieval** shows *why* each passage was retrieved (relevance explanation) and ties every claim in the answer to a numbered source. That helps users, helps you debug retrieval, and makes hallucinations visible.

## How it works

1. Retrieve top-k passages
2. Ask the LLM to explain each passage's relevance in one sentence
3. Number the passages in the prompt
4. Require inline `[n]` citations in the answer
5. Validate that cited numbers exist and map back to sources

```text
 question --> top-3 --> [explain each] --> "why retrieved"
                    --> numbered context --> LLM --> "Claim A [1]. Claim B [3]."
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def explain(query, chunk):
    return llm(f"In one sentence, explain how this passage relates to the question.\n\nQuestion: {query}\n\nPassage: {chunk}")

query = "How do I stop a model from making things up?"
hits = store.search(query, 3)
for n, h in enumerate(hits, 1):
    print(f"[{n}] similarity={h['score']:.3f}")
    print("    why:", explain(query, h["text"]))
    print("    text:", h["text"][:120], "...\n")

Ask for inline `[n]` citations so every claim can be traced to a source chunk.

In [ ]:
numbered = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
print(llm(
    f"Context:\n{numbered}\n\nQuestion: {query}\n\nAnswer using only the context. After each claim add the source number like [1].",
    system=RAG_SYSTEM,
))

## Programmatically verify the citations
A citation like `[7]` when only 3 sources exist is a red flag.

In [ ]:
import re
q = "How do reranking and hybrid search help?"
hits = store.search(q, 4)
numbered = "\n\n".join(f"[{n}] {h['text']}" for n, h in enumerate(hits, 1))
response = llm(f"Context:\n{numbered}\n\nQuestion: {q}\n\nAnswer using only the context. After each claim add the source number like [1].", system=RAG_SYSTEM)
cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", response)})
print(response, "\n")
print("cited sources:", cited, "| valid:", all(1 <= n <= len(hits) for n in cited), "| unused sources:", [n for n in range(1, len(hits) + 1) if n not in cited])

## Show the cited text next to each claim

In [ ]:
for n in cited:
    if 1 <= n <= len(hits):
        print(f"[{n}] score={hits[n - 1]['score']:.3f}  {hits[n - 1]['text'][:200]}\n")

## Faithfulness spot-check
For each cited claim, does the cited passage really support it?

In [ ]:
sentences = [s for s in split_sentences(response) if re.search(r"\[\d+\]", s)]
for s in sentences:
    n = int(re.search(r"\[(\d+)\]", s).group(1))
    if 1 <= n <= len(hits):
        ok = llm_yes(f"Passage: {hits[n - 1]['text']}\n\nClaim: {s}\nDoes the passage support the claim?")
        print("SUPPORTED " if ok else "UNSUPPORTED", "|", s[:110])

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Builds user trust | One extra LLM call per source for explanations |
| Makes retrieval debuggable | Small models sometimes cite wrongly |
| Citations can be machine-verified | Longer outputs |

## When to use it

Any user-facing assistant, especially in regulated or high-trust domains.

## Experiments to try

- Highlight the exact supporting sentence instead of the whole chunk.
- Reject answers whose citations fail verification.
- Show similarity scores as a confidence bar.

## Key takeaways

- Number your sources.
- Verify citations in code.
- Explanations double as debugging output.

---
**Next:** [27. Corrective RAG (CRAG)](../27_crag/crag.ipynb)